# Engenharia de features: criar coluna, e medir se ela paga

O notebook anterior transformou colunas que já existiam. Este faz outra coisa: **cria** colunas que não
estavam na tabela, a partir do que já estava lá.

Feature nova é a parte do trabalho em que a informação de domínio entra, e é também onde mais se gasta
tempo por nada. A regra do notebook é simples e desconfortável: **toda feature criada aqui é medida em
validação cruzada, com e sem ela, no mesmo modelo**. Ganho que não aparece na medição não fica, por
bonita que seja a ideia.

A tabela tem verdade declarada, para a medição significar alguma coisa: o alvo depende de uma
**interação** entre duas colunas, de um **mês do ano**, de uma **renda baixa** e do **contexto da
cidade**. Nenhuma das colunas cruas carrega isso sozinha.


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (KBinsDiscretizer, FunctionTransformer, OneHotEncoder,
                                   QuantileTransformer, StandardScaler)

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (9, 4)})

RANDOM_STATE = 42
ALVO = "cancelou"


def tabela_de_clientes(n_linhas=1500, seed=RANDOM_STATE):
    """Base com verdade declarada: interação, calendário, renda baixa e contexto de cidade.

    - `engajamento_a` e `engajamento_b` só importam **em conjunto**, pelo produto dos dois;
    - `data_de_entrada` importa pelo **mês**, e os meses 3 e 12 pesam mais;
    - `renda_mensal` importa por um **corte**, não em linha reta;
    - cada cidade tem um **traço próprio** de engajamento, que entra no alvo e que a média do grupo
      consegue recuperar sem olhar o alvo;
    - `observacao` importa pela palavra "urgente", presente em 75% de quem cancelou e em 20% de quem
      não cancelou: informativa, e não uma cópia do alvo.
    """
    rng = np.random.default_rng(seed)
    cidades = [f"cidade_{i:02d}" for i in range(1, 26)]
    traco_por_cidade = {cidade: rng.normal(0, 1.0) for cidade in cidades}
    cidade = rng.choice(cidades, n_linhas)
    traco_da_cidade = pd.Series(cidade).map(traco_por_cidade).to_numpy()

    engajamento_a = traco_da_cidade + rng.normal(0, 1.0, n_linhas)
    engajamento_b = rng.normal(0, 1.0, n_linhas)
    renda = rng.lognormal(mean=8.2, sigma=0.6, size=n_linhas).round(2)
    entrada = pd.to_datetime("2024-01-01") + pd.to_timedelta(rng.integers(0, 365, n_linhas), unit="D")

    sinal = (1.10 * (engajamento_a * engajamento_b > 0.2)   # a interação
             + 0.70 * np.isin(entrada.month, [3, 12])       # o mês
             + 0.60 * (renda < 3000)                        # o corte na renda
             + 0.80 * traco_da_cidade                       # o traço da cidade
             + rng.normal(0, 1.0, n_linhas))
    cancelou = (sinal > np.quantile(sinal, 0.80)).astype(int)

    urgente = rng.random(n_linhas) < np.where(cancelou == 1, 0.75, 0.20)
    base = pd.DataFrame({
        "engajamento_a": engajamento_a.round(3),
        "engajamento_b": engajamento_b.round(3),
        "renda_mensal": renda,
        "data_de_entrada": entrada,
        "cidade": cidade,
        "observacao": np.where(urgente, "pedido com atraso urgente", "pedido sem detalhe"),
        ALVO: cancelou,
    })
    # o texto varia de tamanho, e o tamanho também é usado como feature depois
    base["observacao"] = base["observacao"] + np.where(rng.random(n_linhas) < 0.5, " e detalhes", "")
    return base


tabela = tabela_de_clientes()
CRUAS = ["engajamento_a", "engajamento_b", "renda_mensal"]

print(f"tabela: {tabela.shape[0]} linhas x {tabela.shape[1]} colunas")
print(f"alvo: {tabela[ALVO].mean():.1%} de cancelamento")
print()
print(tabela.head(5).to_string())


## O alvo depende de uma interação

O desenho da verdade: quem usa muito os **dois** produtos cancela, e quem usa muito um só, não. É o
efeito conjunto, não a coluna. Olhe o que a correlação de cada coluna crua diz sobre o alvo, e compare
com o que o produto das duas diz.


In [ ]:
correlacoes = {
    "engajamento_a": tabela["engajamento_a"].corr(tabela[ALVO]),
    "engajamento_b": tabela["engajamento_b"].corr(tabela[ALVO]),
    "renda_mensal": tabela["renda_mensal"].corr(tabela[ALVO]),
    "engajamento_a * engajamento_b": (tabela["engajamento_a"] * tabela["engajamento_b"]).corr(tabela[ALVO]),
    "(engajamento_a * engajamento_b) > 0.2": (
        (tabela["engajamento_a"] * tabela["engajamento_b"]) > 0.2).astype(int).corr(tabela[ALVO]),
}
print("correlacao com o alvo:")
for nome, valor in correlacoes.items():
    print(f"  {nome:40s} {valor: .3f}")

fig, eixos = plt.subplots(1, 2, figsize=(13, 3.6))
sns.scatterplot(tabela.sample(500, random_state=RANDOM_STATE), x="engajamento_a", y="engajamento_b",
                hue=ALVO, alpha=0.6, s=18, ax=eixos[0])
eixos[0].set_title("cru: dois produtos")
produto = tabela["engajamento_a"] * tabela["engajamento_b"]
sns.boxplot(x=(produto > 0.2), y=tabela[ALVO], ax=eixos[1])
eixos[1].set_title("o produto das duas, cortado em 0.2")
eixos[1].set_xlabel("produto acima de 0.2")
plt.tight_layout()
plt.show()

print()
print("taxa de cancelamento por produto acima do corte:")
print(tabela.groupby((produto > 0.2).rename("produto > 0.2"))[ALVO].agg(["mean", "size"]).round(3).to_string())


`engajamento_b` sozinho quase não diz nada (correlação 0,029), `engajamento_a` diz um pouco (0,186,
porque carrega junto o traço da cidade) e o **produto cortado em 0,2** diz o resto (0,319). Na prática:
10,7% de cancelamento abaixo do corte, 37,4% acima.

A mesma tabela organizada de outro jeito muda o que o modelo consegue ver. E vale registrar o que a
medição mostrou e a intuição não: nesta base a árvore **não** levou vantagem sobre a regressão logística
antes das features criadas (0,745 contra 0,737). O gargalo não era o modelo, era o que faltava na
tabela, e é disso que o resto do notebook trata.

## Medindo cada feature criada

Daqui para frente, tudo passa pela mesma régua: mesma validação cruzada, mesmo modelo, mesmo alvo. O
modelo base é a regressão logística, que é quem mais sofre com efeito não linear. No fim há a
comparação com uma árvore.


In [ ]:
def auc_medio(pipeline, X, y, folds=5):
    cv = StratifiedKFold(folds, shuffle=True, random_state=RANDOM_STATE)
    notas = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc")
    return notas.mean(), notas.std()


def monta(colunas, transformacao=None):
    passos = [("escala", StandardScaler())]
    if transformacao is not None:
        passos.insert(0, ("transforma", transformacao))
    return Pipeline([("prepara", ColumnTransformer([
        ("num", Pipeline(passos), colunas),
    ], remainder="drop")), ("modelo", LogisticRegression(max_iter=5000))])


y = tabela[ALVO]
linhas = []

media, desvio = auc_medio(monta(CRUAS), tabela, y)
linhas.append({"feature criada": "nenhuma (colunas cruas)", "roc_auc": media, "desvio": desvio})

com_interacao = tabela.copy()
com_interacao["engajamento_a_x_b"] = com_interacao["engajamento_a"] * com_interacao["engajamento_b"]
media, desvio = auc_medio(monta(CRUAS + ["engajamento_a_x_b"]), com_interacao, y)
linhas.append({"feature criada": "produto dos engajamentos", "roc_auc": media, "desvio": desvio})

com_razao = com_interacao.copy()
com_razao["razao_entre_engajamentos"] = (com_razao["engajamento_a"] / com_razao["engajamento_b"].abs().clip(lower=0.05))
media, desvio = auc_medio(monta(CRUAS + ["engajamento_a_x_b", "razao_entre_engajamentos"]), com_razao, y)
linhas.append({"feature criada": "produto + razão entre eles", "roc_auc": media, "desvio": desvio})

display(pd.DataFrame(linhas).set_index("feature criada").round(4))


O produto resolve. A razão, que é a feature que **não** estava na verdade do problema, não acrescenta
nada: ela aparece na tabela com o mesmo número de antes, dentro do desvio. Criar feature por analogia,
sem hipótese, é o jeito mais rápido de encher a tabela de coluna inútil.

## Data: a coluna que o modelo não lê

Uma coluna de data é inútil do jeito que está: modelo não multiplica `2024-07-14`. O que serve é o que
a data **é** em termos de calendário, e o mês é o caso clássico (sazonalidade).


In [ ]:
com_data = com_interacao.copy()
com_data["mes"] = com_data["data_de_entrada"].dt.month
com_data["dia_da_semana"] = com_data["data_de_entrada"].dt.dayofweek
com_data["fim_de_semana"] = (com_data["dia_da_semana"] >= 5).astype(int)

print("taxa de cancelamento por mes:")
print(com_data.groupby("mes")[ALVO].agg(["mean", "size"]).round(3).to_string())

media, desvio = auc_medio(monta(CRUAS + ["engajamento_a_x_b", "mes", "dia_da_semana", "fim_de_semana"]),
                          com_data, y)
linhas.append({"feature criada": "produto + mês como número, dia da semana", "roc_auc": media,
               "desvio": desvio})

# o mes como categoria, e nao como numero: o efeito esta em meses especificos
mes_em_colunas = pd.get_dummies(com_data["mes"], prefix="mes").astype(int)
com_data = pd.concat([com_data, mes_em_colunas], axis=1)
media, desvio = auc_medio(monta(CRUAS + ["engajamento_a_x_b"] + list(mes_em_colunas.columns)),
                          com_data, y)
linhas.append({"feature criada": "produto + mês em one-hot", "roc_auc": media, "desvio": desvio})
display(pd.DataFrame(linhas).set_index("feature criada").round(4))


A tabela mostra duas coisas que valem mais que a feature em si. Primeiro: o mês importa de verdade
(março e dezembro cancelam 36%, contra 20% na média dos meses, quase o dobro), mas **mês como número não
entrega isso**, porque o efeito não é crescente ao longo do ano. Como número, ele até piorou o resultado
(0,710 contra 0,716 do produto sozinho); como categoria, subiu para 0,732. A coluna estava certa e a
representação estava errada, que é a mesma lição do notebook de encoding, aparecendo no meio da
engenharia de features.

Segundo: `dia_da_semana` e `fim_de_semana` entraram sem efeito nenhum. Custo de calcular: nenhum. Custo
de manter: alguém vai ter de entender, documentar e carregar essas colunas para sempre.

## Duas armadilhas de data que valem a regra

- **data que só existe depois do desfecho** é vazamento, como `data_do_cancelamento`;
- **data crua como número** (transformar em timestamp) ensina o modelo a decorar o período do treino, e
  ele erra em qualquer período fora dele.

## Distribuição torta: transformar, cortar, ou nenhum dos dois

`renda_mensal` tem cauda longa, e na verdade do problema ela importa por um **corte** (abaixo de 3000).
Três candidatas: `log`, que comprime a cauda; `QuantileTransformer`, que força a distribuição a ficar
uniforme; e `KBinsDiscretizer`, que troca a coluna por faixas.

Para a comparação valer, cada uma entra **no lugar** da coluna crua, e não ao lado dela. Colocar a
transformação ao lado é o erro que faz a tabela mentir: o modelo continua vendo a coluna original.


In [ ]:
sem_efeito = com_data.copy()
sem_renda = [c for c in CRUAS + ["engajamento_a_x_b"] if c != "renda_mensal"]


def monta_com_renda(transformacao=None):
    """Compara a renda crua com uma versao dela, no mesmo lugar da tabela."""
    ramos = [("resto", Pipeline([("escala", StandardScaler())]), sem_renda)]
    if transformacao is None:
        ramos.insert(0, ("renda", Pipeline([("escala", StandardScaler())]), ["renda_mensal"]))
    else:
        ramos.insert(0, ("renda", Pipeline([("transforma", transformacao),
                                            ("escala", StandardScaler())]), ["renda_mensal"]))
    return Pipeline([("prepara", ColumnTransformer(ramos, remainder="drop")),
                     ("modelo", LogisticRegression(max_iter=5000))])


media, desvio = auc_medio(monta_com_renda(FunctionTransformer(np.log1p)), sem_efeito, y)
linhas.append({"feature criada": "produto + log no lugar da renda crua", "roc_auc": media,
               "desvio": desvio})

media, desvio = auc_medio(monta_com_renda(QuantileTransformer(n_quantiles=100,
                                                              output_distribution="normal",
                                                              random_state=RANDOM_STATE)),
                          sem_efeito, y)
linhas.append({"feature criada": "produto + quantile no lugar da renda crua", "roc_auc": media,
               "desvio": desvio})

com_faixas = com_data.copy()
faixas = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile")
com_faixas["renda_em_faixas"] = faixas.fit_transform(com_faixas[["renda_mensal"]]).ravel()
media, desvio = auc_medio(monta(sem_renda + ["renda_em_faixas"]), com_faixas, y)
linhas.append({"feature criada": "produto + renda em 5 faixas no lugar da crua", "roc_auc": media,
               "desvio": desvio})

fig, eixos = plt.subplots(1, 2, figsize=(13, 3.2))
sns.histplot(com_faixas["renda_mensal"], kde=True, ax=eixos[0])
eixos[0].set_title("renda como está (assimetria alta)")
sns.histplot(np.log1p(com_faixas["renda_mensal"]), kde=True, ax=eixos[1])
eixos[1].set_title("depois do log")
plt.tight_layout()
plt.show()

display(pd.DataFrame(linhas).set_index("feature criada").round(4))


As três empataram: 0,7202 com log, 0,7199 com quantile e 0,7200 com faixas, contra 0,7156 do produto
sozinho. Diferença dentro do desvio nas três, ou seja, **nenhuma das três se pagou** nesta tabela.

O ponto não é o resultado, é o método: transformar coluna numérica por reflexo é tão ruim quanto nunca
transformar. Aqui a intuição dizia que a cauda longa atrapalhava, e a medição disse que não era por ali
que o modelo estava perdendo.

## Agregação por grupo: a informação que a linha sozinha não tem

`cidade` é categórica, e a tentação é codificá-la e pronto. O que interessa nela aqui é o **comportamento
médio do grupo**, que é uma coluna nova, numérica, calculável sem olhar o alvo.

O cuidado: agregação calculada em cima de toda a tabela usa as linhas de validação no cálculo. O certo é
calcular no treino e aplicar ao resto, e é isso que o transformador abaixo faz, dentro do pipeline.


In [ ]:
class AgregacaoDeGrupo(BaseEstimator, TransformerMixin):
    """Troca cada linha pelo comportamento medio do grupo dela, calculado no treino.

    Nao olha o alvo: agrega as colunas numericas por grupo. O que ele traz e contexto, nao resposta.
    """

    def __init__(self, grupo="cidade", colunas=("engajamento_a",), sufixo="_media_do_grupo"):
        self.grupo = grupo
        self.colunas = colunas
        self.sufixo = sufixo

    def fit(self, X, y=None):
        self.medias_ = X.groupby(self.grupo)[list(self.colunas)].mean()
        self.geral_ = X[list(self.colunas)].mean()
        return self

    def transform(self, X):
        nomes = [f"{c}{self.sufixo}" for c in self.colunas]
        medias = self.medias_.copy()
        medias.columns = nomes
        saida = X[[self.grupo]].merge(medias, left_on=self.grupo, right_index=True, how="left")[nomes]
        preenchimento = {nome: valor for nome, valor in zip(nomes, self.geral_.to_numpy())}
        return saida.fillna(preenchimento).to_numpy()


agregacao = Pipeline([
    ("agrega", AgregacaoDeGrupo(grupo="cidade", colunas=("engajamento_a",))),
    ("escala", StandardScaler()),
])
pipeline_com_agregacao = Pipeline([
    ("prepara", ColumnTransformer([
        ("num", Pipeline([("escala", StandardScaler())]), CRUAS + ["engajamento_a_x_b"]),
        ("grupo", agregacao, ["cidade", "engajamento_a"]),
    ], remainder="drop")),
    ("modelo", LogisticRegression(max_iter=5000)),
])
media, desvio = auc_medio(pipeline_com_agregacao, com_data, y)
linhas.append({"feature criada": "produto + média do grupo (no pipeline)", "roc_auc": media,
               "desvio": desvio})
display(pd.DataFrame(linhas).set_index("feature criada").round(4))

print()
print("o comportamento do grupo, por cidade (calculado no treino inteiro, so para olhar):")
print(com_data.groupby("cidade")[["engajamento_a", ALVO]].mean().round(3).head(8).to_string())


A média de engajamento por cidade foi o maior ganho isolado do notebook: 0,784 contra 0,716 do produto
sozinho, e sem olhar o alvo em momento nenhum. Ela acrescenta o que a linha não tem, que é o contexto em
que aquela linha está.

O cuidado aqui é **onde** ela é calculada, não se ela pode ser calculada. Um transformador que calcula as
médias por grupo dentro do `fit` aprende só com o treino de cada fold, e é isso que o `AgregacaoDeGrupo`
faz. A alternativa, calcular a média na tabela inteira e depois dividir, entrega ao modelo um pedaço das
linhas de validação.

Quando a agregação envolve o alvo (média de cancelamento por cidade, por exemplo), o nome muda e o
cuidado cresce: aquilo é codificação pelo alvo, e a lição do notebook anterior se aplica inteira.

## Texto: contagem e palavra-chave

Coluna de texto livre é o caso em que criar feature é mais barato do que parece. Antes de partir para
modelo de linguagem, mede-se o básico: tamanho, e presença de palavra que importa.


In [ ]:
com_texto = com_data.copy()
com_texto["observacao_tamanho"] = com_texto["observacao"].str.len()
com_texto["observacao_palavras"] = com_texto["observacao"].str.split().str.len()
com_texto["menciona_urgente"] = com_texto["observacao"].str.contains("urgente").astype(int)

print("taxa de cancelamento por menção a urgência:")
print(com_texto.groupby("menciona_urgente")[ALVO].agg(["mean", "size"]).round(3).to_string())

media, desvio = auc_medio(monta(CRUAS + ["engajamento_a_x_b", "observacao_tamanho",
                                         "observacao_palavras", "menciona_urgente"]), com_texto, y)
linhas.append({"feature criada": "produto + tamanho e palavra-chave do texto", "roc_auc": media,
               "desvio": desvio})
display(pd.DataFrame(linhas).set_index("feature criada").round(4))


Tamanho do texto e menção à palavra que importa resolvem o essencial, e custam três linhas de código:
0,844 contra 0,716, o maior salto isolado do notebook. E dá para ver por quê nos números: 7,5% de
cancelamento em quem não menciona urgência, 48,3% em quem menciona.

## Tudo junto, e a mesma coluna em outro modelo

A tabela abaixo compara o modelo linear com uma árvore, nas mesmas features. A diferença entre as duas
colunas é a resposta para "vale a pena criar esta feature?": depende de quem vai usá-la.


In [ ]:
construidas = (["engajamento_a_x_b", "mes", "dia_da_semana", "fim_de_semana"]
               + list(mes_em_colunas.columns)
               + ["renda_em_faixas", "observacao_tamanho", "observacao_palavras", "menciona_urgente"])
tabela_final = com_data.copy()
tabela_final["renda_em_faixas"] = com_faixas["renda_em_faixas"]
tabela_final["observacao_tamanho"] = com_texto["observacao_tamanho"]
tabela_final["observacao_palavras"] = com_texto["observacao_palavras"]
tabela_final["menciona_urgente"] = com_texto["menciona_urgente"]

resultado = []
for nome, classificador in {
    "regressao logistica": LogisticRegression(max_iter=5000),
    "gradient boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}.items():
    for variante, colunas, usa_agregacao in [
        ("cruas", CRUAS + ["cidade"], False),
        ("cruas + features criadas (com a media do grupo)", CRUAS + ["cidade"] + construidas, True),
    ]:
        ramos = [
            ("num", Pipeline([("escala", StandardScaler())]),
             [c for c in CRUAS + construidas if c in colunas]),
            ("cidade", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["cidade"]),
        ]
        if usa_agregacao:
            ramos.append(("grupo", agregacao, ["cidade", "engajamento_a"]))
        pipeline = Pipeline([("prepara", ColumnTransformer(ramos, remainder="drop")),
                             ("modelo", classificador)])
        media, desvio = auc_medio(pipeline, tabela_final, y)
        resultado.append({"modelo": nome, "features": variante, "roc_auc": round(media, 4),
                          "desvio": round(desvio, 4)})

display(pd.DataFrame(resultado).set_index(["modelo", "features"]))


## O resultado junto

Com as features criadas, os dois modelos empataram de novo, 0,881 na regressão logística e 0,880 no
boosting, e os dois subiram quase 0,14 de AUC em relação às colunas cruas (0,737 e 0,745). Nesta tabela,
criar coluna valeu mais do que escolher modelo, e isso é comum em dado tabular: o modelo não inventa
informação que não está lá de forma utilizável.

## Como ler isso

O roteiro para criar feature sem se perder:

1. escreva a **hipótese** antes do código: o que você acha que o modelo não está vendo?
2. crie a coluna e **meça com e sem ela**, na mesma validação cruzada e no mesmo modelo;
3. se o ganho estiver dentro do desvio, a feature não se paga, e o custo dela é manutenção e confusão
   para quem ler a tabela depois;
4. lembre que a feature pertence ao **par** com o modelo: o que ajuda a regressão logística pode não
   ajudar a árvore;
5. para cada coluna criada, pergunte **quando** ela passa a existir, do mesmo jeito que se pergunta
   para a coluna original.

## Resumo

Engenharia de features é hipótese e medição, nesta ordem. Neste notebook: o produto de duas colunas
resolve um efeito conjunto que nenhuma delas carrega sozinha (0,716 contra 0,659); a data vira mês, e o
mês **como categoria**, não como número, leva o resultado a 0,732; a média de engajamento do grupo traz o
contexto que a linha não tem, e faz o maior ganho isolado, 0,784; e o texto entrega o essencial em três
linhas, com tamanho e palavra-chave, chegando a 0,844.

E fica o que a medição recusou: a razão entre as duas colunas de engajamento, o log da renda, o quantile
e as faixas empataram com o que já havia. Quatro ideias boas no papel, todas descartadas pela mesma
régua. Com tudo junto, os modelos empataram em ~0,88, o que resume o notebook: o trabalho estava na
tabela, não na escolha do modelo.

**Próximo passo:** o notebook de validação cruzada desta pasta, que é a régua que este notebook usou
para decidir tudo.

## Referências

```bibtex
@inproceedings{heaton2016empirical,
  title     = {An Empirical Analysis of Feature Engineering for Predictive Modeling},
  author    = {Heaton, Jeff},
  booktitle = {SoutheastCon 2016},
  pages     = {1--6},
  year      = {2016},
  doi       = {10.1109/SECON.2016.7506650}
}

@book{zheng2018feature,
  title     = {Feature Engineering for Machine Learning: Principles and Techniques for Data Scientists},
  author    = {Zheng, Alice and Casari, Amanda},
  year      = {2018},
  publisher = {O'Reilly Media}
}
```
